In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("intro2LinReg.ipynb")

## Lecture Section

In this lecture, we will cover the `statsmodels` library.
We will cover:
* What is linear regression?
    * The OLS method & residuals
    * Simple vs. multiple regression
* Visualizing your data
* Regression with `statsmodels`
    * confidence & prediction intervals
    * predictions
    * Durbin-Watson test


We are going to use the `tips` dataset again.

In [ ]:
import matplotlib.pyplot as plt
import numpy.random
import seaborn as sns
import numpy as np
from proto import STRING
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
import pandas as pd

tips = sns.load_dataset("tips")
tips.head()


### What is Linear Regression?

Linear regression is one of the most foundational tools in statistics and data science. At its core, it answers a simple question: **can we draw a line through our data that best describes the relationship between two variables?**

For example, imagine we're looking at restaurant bills and tips. We might suspect that larger bills tend to produce larger tips. Linear regression lets us formally describe that relationship — and then use it to make predictions.

The general form of a simple linear regression model is:

$$y = \beta_0 + \beta_1 x + \varepsilon$$

Where:
- $y$ is the **dependent variable** (what we're trying to predict — e.g., tip)
- $x$ is the **independent variable** (what we're using to predict — e.g., total bill)
- $\beta_0$ is the **intercept** — the predicted value of $y$ when $x = 0$
- $\beta_1$ is the **slope** — how much $y$ changes for a one-unit increase in $x$
- $\varepsilon$ is the **error term** — the part of $y$ that our line can't explain

The goal is to find the values of $\beta_0$ and $\beta_1$ that make our line fit the data as well as possible.

### Minimizing Residuals

How do we decide which line is "best"? One answer is the **Ordinary Least Squares (OLS)** method — the same method that `statsmodels` uses.

For each data point, the **residual** is the vertical distance between the actual value and the value predicted by our line:

$$\text{residual}_i = y_i - \hat{y}_i$$

OLS finds the line that **minimizes the sum of the squared residuals** (SSR):

$$\text{SSR} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

We square the residuals so that positive and negative errors don't cancel out, and so that large errors are penalized more heavily.

The plot below illustrates this — the red dashed lines are the residuals for each point.

In [ ]:
# Visualizing what OLS is minimizing
np.random.seed(42)
x_demo = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
y_demo = 1.5 * x_demo + 2 + np.random.normal(0, 1.5, size=len(x_demo))

# Fit a line manually using numpy
coeffs = np.polyfit(x_demo, y_demo, 1)
y_hat = np.polyval(coeffs, x_demo)

plt.figure(figsize=(8, 5))
plt.scatter(x_demo, y_demo, color='steelblue', zorder=5, label='Data points')
plt.plot(x_demo, y_hat, color='black', label=f'Fitted line: y = {coeffs[0]:.2f}x + {coeffs[1]:.2f}')

# Draw residuals
for xi, yi, yhi in zip(x_demo, y_demo, y_hat):
    plt.plot([xi, xi], [yi, yhi], color='red', linestyle='--', linewidth=1)

plt.title('OLS: Minimizing the Sum of Squared Residuals')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.tight_layout()
plt.show()

### Simple vs. Multiple Linear Regression

The example above only uses **one predictor** ($x$) — this is called **simple linear regression**.

In practice, we often want to use **multiple predictors** at the same time. For example, we might want to predict tip using both total bill *and* the size of the dining party. This is called **multiple linear regression**:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \varepsilon$$

`statsmodels` handles both cases with the same syntax — you just add more terms to the formula string. We will cover multiple regression in more detail in two lectures.

### Visualizing the Data First

Before fitting any model, it's always a good idea to **look at your data**. A scatter plot can tell you a lot about whether a linear relationship is even a reasonable assumption.

Let's look at the relationship between `total_bill` and `tip` in the `tips` dataset.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=tips, x='total_bill', y='tip', alpha=0.6)
plt.title('Total Bill vs. Tip')
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip ($)')
plt.tight_layout()
plt.show()

There does appear to be a positive relationship — as the bill goes up, so does the tip. It's not a perfect line, but linear regression can help us describe the general trend and quantify the relationship.

Let's also look at a `regplot`, which overlays a regression line and confidence band directly on the scatter plot:

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=tips, x='total_bill', y='tip', scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
plt.title('Total Bill vs. Tip with Regression Line')
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip ($)')
plt.tight_layout()
plt.show()

Now we have a good visual sense of what we're modeling. Let's fit the regression formally using `statsmodels`.

### Regression

First, we load in the library that contains the linear regression function: `statsmodels.formula.api` - I renamed it to `smf`. This differs from `import statsmodels.api as sm` - the `formula` version uses R-style formulation (it automatically includes the intercept, for example). If you use the other, you'll need to add the intercept yourself with `.add_constant()`.

Next, we use the alias with the `.ols()` function! It takes the equation and our data, then we `.fit()` our data to the model.

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols("tip ~ total_bill", data=tips).fit()
print(model.summary())

In [ ]:
# ### Plot residuals
residuals = model.resid
fitted = model.fittedvalues
sns.residplot(x=fitted, y=residuals, lowess=True)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residual Plot")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.show()

It gives us a lot of information - really similar to R output. We can see the intercept (0.9203), slope (0.1050), and some statistics like the Durbin-Watson test (2.151).

If you recall from last lecture, there is some heteroscedasticity. Let's re-run with a `.log()` around our dependant variable.

In [ ]:
model_log = smf.ols("np.log(tip) ~ total_bill", data=tips).fit()
print(model_log.summary())

In [ ]:
residuals = model_log.resid
fitted = model_log.fittedvalues
sns.residplot(x=fitted, y=residuals, lowess=True)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residual Plot")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.show()

Not perfect, but it fits a bit better! I'll explain what these plots mean in the next lecture; however, the more even spread around the red-line, the better!

#### Confidence & Prediction Intervals

With `statsmodels`, it's very easy to get confidence and prediction intervals.

* Confidence interval: range where the true population is likely to belong. If we compute a 95% confidence interval for each of 100 samples, we would expect roughly 95 of those confidence intervals to contain the true parameter value.
* Prediction interval: range that a new observation might fall in. It is always wider than the confidence interval becauce it accounts for more variability.

To get the confidence intervals of the model parameters, we just call our model with `.conf_int()`.

* We will use our non-logged model - it's a little easier to interpret.

In [ ]:
model.conf_int()

To get the cnfidence and rediction intervals of our fitted values, we first call `.get_prediction()` with our value. Then, we call `summary_frame()` with our desired `alpha` value (in our case, we set to 0.05 for a 95% interval). `mean_ci_lower` and `mean_ci_upper` are the confidence intervals, and `obs_ci_lower` and `obs_ci_upper` is our prediction interval.

In [ ]:
pred = model.get_prediction({"total_bill": [20]})
pred_summary = pred.summary_frame(alpha=0.05)
print(pred_summary)

For fun, let's predict tips for a bunch of bills and plot our confidence and prediction intervals.

In [ ]:
# Create DataFrame of new X values
x_vals = np.linspace(tips["total_bill"].min(), tips["total_bill"].max(), 100)
new_data = pd.DataFrame({"total_bill": x_vals})
preds = model.get_prediction(new_data).summary_frame()

# Plot with correct layering and values
plt.figure(figsize=(10, 6))
plt.rcParams['axes.facecolor'] = 'snow'
plt.scatter(tips["total_bill"], tips["tip"], alpha=0.5, label="Observed data")
plt.plot(x_vals, preds["mean"], color="blue", label="Regression line")
plt.fill_between(x_vals, preds["mean_ci_lower"], preds["mean_ci_upper"],
                 color="lightskyblue", alpha=0.5, label="95% CI (mean)")
plt.fill_between(x_vals, preds["obs_ci_lower"], preds["obs_ci_upper"],
                 color="lightcoral", alpha=0.3, label="95% Prediction Interval")
plt.xlabel("Total Bill")
plt.ylabel("Tip")
plt.title("Confidence and Prediction Intervals for Tip ~ Total Bill")
plt.legend()
plt.show()


#### Durbin-Watson Test

Using the R-formula linear regression option gives us the Durbin-Watson test automatically. We can also ask for it separately. It isn't very helpful for this model, but I want to show the syntax anyway. If you use a lot of different tests, there's a good chance it's already available in `statsmodels`:

https://www.statsmodels.org/stable/stats.html#module-statsmodels.stats.stattools

In [ ]:
durbin_watson(model.resid)

**Question 1.**

Use the `diamonds` dataset from seaborn.
Fit a simple OLS regression model predicting `price` from `carat` using `statsmodels`, then extract key results.

- `model`: the fitted OLS model using the formula `"price ~ carat"`
- `slope`: the estimated coefficient (slope) for `carat`
- `intercept`: the estimated intercept

In [ ]:
import random
import statsmodels.formula.api as smf
diamonds = sns.load_dataset("diamonds")
diamonds = diamonds.sample(150)

model = ...
slope = ...
intercept = ...

In [ ]:
grader.check("q1")

**Question 2.**

Using the `model` from Question 1, compute the following:

- `residuals`: the residuals from the model (as a pandas Series)
- `ssr`: the sum of squared residuals (a single float)
- `dw_stat`: the Durbin-Watson statistic for the model residuals

In [ ]:
import numpy as np
model = ...
residuals = ...
ssr = ...
dw_stat = ...

In [ ]:
grader.check("q2")

**Question 3.**

There are other ways to minimize residuals. Search for a method other than OLS, and writ ea quick paragraph summarising how it works and why you might use it over OLS. The paragraph should be four sentences or longer.


In [ ]:
 ans = ...

In [ ]:
grader.check("q3")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()